In [49]:
import re

def preprocess_text(text):
    text = text.lower() # lower case
    text = re.sub(r"[^a-z\s]", "", text)  # remove punctuation
    tokens = text.split() # splitting at spaces
    return tokens

In [50]:
dataset = [
    ("go left", "MOVE_LEFT"),
    ("move left", "MOVE_LEFT"),
    ("left", "MOVE_LEFT"),
    ("back left", "MOVE_LEFT"),
    ("walk left", "MOVE_LEFT"),
    ("backward", "MOVE_LEFT"),

    ("go right", "MOVE_RIGHT"),
    ("move right", "MOVE_RIGHT"),
    ("right", "MOVE_RIGHT"),
    ("forward", "MOVE_RIGHT"),
    ("move ahead", "MOVE_RIGHT"),

    ("jump", "JUMP"),
    ("hop", "JUMP"),
    ("leap", "JUMP"),
    ("up", "JUMP"),
    ("jump now", "JUMP"),

    ("run", "RUN"),
    ("sprint", "RUN"),
    ("go fast", "RUN"),
    ("speed up", "RUN"),
    ("full speed", "RUN"),

    ("duck", "DUCK"),
    ("crouch", "DUCK"),
    ("go low", "DUCK"),
    ("get down", "DUCK"),
    ("bend down", "DUCK"),

    ("fire", "FIRE"),
    ("shoot", "FIRE"),
    ("throw fireball", "FIRE"),
    ("blast", "FIRE"),
    ("attack", "FIRE"),

    ("stop", "STOP"),
    ("freeze", "STOP"),
    ("hold still", "STOP"),
    ("don't move", "STOP"),
    ("stay", "STOP"),

    ("pause", "PAUSE"),
    ("stop game", "PAUSE"),
    ("pause game", "PAUSE"),
    ("take a break", "PAUSE"),
    ("halt", "PAUSE"),

    ("go down", "ENTER_PIPE"),
    ("enter pipe", "ENTER_PIPE"),
    ("drop down", "ENTER_PIPE"),
    ("go inside", "ENTER_PIPE"),
    ("down pipe", "ENTER_PIPE"),
]

In [51]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

In [52]:
texts = [x[0] for x in dataset]
labels = [x[1] for x in dataset]

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(texts)

In [53]:
model = LogisticRegression()
model.fit(X, labels)

LogisticRegression()

In [54]:
def predict(text):
    x = vectorizer.transform([text])

    probs = model.predict_proba(x)[0]   # probabilities for each class
    best_idx = probs.argmax()           # index of highest probability

    label = model.classes_[best_idx]    # corresponding intent
    confidence = probs[best_idx]        # probability score

    return label, confidence

In [55]:
# testing loop

correct = 0

for text, label in dataset:
    pred = predict(text)
    print(text, "→", pred, "| expected:", label)

    if pred == label:
        correct += 1

accuracy = correct / len(dataset)
print("Accuracy:", accuracy)

go left → (np.str_('MOVE_LEFT'), np.float64(0.40170810179322475)) | expected: MOVE_LEFT
move left → (np.str_('MOVE_LEFT'), np.float64(0.4031008164729919)) | expected: MOVE_LEFT
left → (np.str_('MOVE_LEFT'), np.float64(0.5350132989109412)) | expected: MOVE_LEFT
back left → (np.str_('MOVE_LEFT'), np.float64(0.4135354891089415)) | expected: MOVE_LEFT
walk left → (np.str_('MOVE_LEFT'), np.float64(0.4135354891089415)) | expected: MOVE_LEFT
backward → (np.str_('MOVE_LEFT'), np.float64(0.22463165974013471)) | expected: MOVE_LEFT
go right → (np.str_('MOVE_RIGHT'), np.float64(0.3212983855146571)) | expected: MOVE_RIGHT
move right → (np.str_('MOVE_RIGHT'), np.float64(0.40637626798007914)) | expected: MOVE_RIGHT
right → (np.str_('MOVE_RIGHT'), np.float64(0.3911534358143178)) | expected: MOVE_RIGHT
forward → (np.str_('MOVE_RIGHT'), np.float64(0.21187406819649862)) | expected: MOVE_RIGHT
move ahead → (np.str_('MOVE_RIGHT'), np.float64(0.2350568626176011)) | expected: MOVE_RIGHT
jump → (np.str_('JUM

In [56]:
# longer test cases

test_cases = [
    "go backward left",
    "jump up high",
    "move super fast",
    "shoot him",
    "pause the game",
    "drop into pipe"
]

for t in test_cases:
    print(t, "→", predict(t))

go backward left → (np.str_('MOVE_LEFT'), np.float64(0.43203937222932426))
jump up high → (np.str_('JUMP'), np.float64(0.3534602338910968))
move super fast → (np.str_('RUN'), np.float64(0.16408110519807545))
shoot him → (np.str_('FIRE'), np.float64(0.2451023346132632))
pause the game → (np.str_('PAUSE'), np.float64(0.39429903021923757))
drop into pipe → (np.str_('ENTER_PIPE'), np.float64(0.2715645692941372))
